# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saisathwik2703/flyrank.ai_internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content page (URL/article), aggregated over a trailing 90-day window ending at export time.**

There is no explicit date column in the dataset — all metrics are already aggregated. The 90-day window is defined in the data dictionary: every row's activity totals cover the most recent 90 days of the page's existence. All rows in this slice are from pages that are at least 90 days old (`content_age_days >= 90`).

Within that 90-day window, two sub-windows are compared:
- **last_30d**: the most recent 30 days (days 61–90 of the window)
- **prev_30d**: the preceding 30 days (days 31–60 of the window)

These sub-windows produce the trend label.

In [1]:
import pandas as pd
import numpy as np
import os

if os.path.exists('data/raw/content_refresh_anonymized.csv'):
    df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
else:
    df = pd.read_csv('../data/raw/content_refresh_anonymized.csv')

# Verify unit of analysis: each content_id should appear exactly once
assert df['content_id'].nunique() == len(df), 'FAIL: duplicate content_ids found'
print(f'Unit check PASSED: {len(df):,} rows, {df["content_id"].nunique():,} unique content_ids')

# Verify time window assumption
print(f'\ncontent_age_days stats:')
print(df['content_age_days'].describe())
print(f'All pages >= 90 days: {(df["content_age_days"] >= 90).all()}')

Unit check PASSED: 30,000 rows, 30,000 unique content_ids

content_age_days stats:
count    30000.00000
mean       256.16780
std        132.70793
min         90.00000
25%        132.00000
50%        236.00000
75%        333.00000
max        564.00000
Name: content_age_days, dtype: float64
All pages >= 90 days: True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Label (target to predict)
| Field | Notes |
|---|---|
| `trend_direction` | Source column used to compute `is_declining_label`. **Never a feature.** |
| `trend_pct` | Quantifies the trend magnitude. **Never a feature** — leaks the label. |
| `is_declining_label` | Engineered binary target: `trend_direction == 'down'` |

### Features (model inputs)
**Numeric:**
| Field | Notes |
|---|---|
| `search_volume` | Keyword opportunity size |
| `competition` | Keyword difficulty (0–1) |
| `cpc` | Keyword commercial value |
| `word_count` | Content length signal |
| `char_count` | Correlated with word_count; include both |
| `log_impressions_90d` | Log-transformed (heavy right tail) |
| `log_clicks_90d` | Log-transformed |
| `log_sessions_90d` | Log-transformed |
| `log_ai_sessions_90d` | Log-transformed; many zeros |
| `days_with_impressions` | Page consistency in search |
| `days_with_sessions` | Page consistency in GA4 |
| `content_age_days` | Staleness risk factor |
| `days_since_last_update` | Freshness: high = likely stale |
| `ctr` | Ratio of clicks to impressions (×100 = %) |
| `avg_position` | Google ranking position |
| `engagement_rate` | Engaged sessions / sessions (×100 = %) |
| `scroll_rate` | Scroll events / pageviews (×100 = %) |
| `ai_traffic_pct` | Share of traffic from AI referrals (×100 = %) |

**Categorical:**
| Field | Notes |
|---|---|
| `competition_level` | LOW / MEDIUM / HIGH |
| `content_type` | keyword article / feedly article / comparison article |
| `main_intent` | informational / transactional / commercial / navigational |
| `age_tier` | Binned age grouping |
| `freshness_tier` | Binned recency grouping |
| `word_count_tier` | Binned length grouping |
| `impression_tier` | Binned traffic volume |
| `position_tier` | Binned ranking band |

### Context (joins/grouping only, not features)
| Field | Notes |
|---|---|
| `content_id` | Unique page identifier — joins only |
| `client_id` | Used to construct client-holdout split |

### Excluded (with reason)
| Field | Reason for exclusion |
|---|---|
| `trend_direction` | Direct source of the label — target leakage |
| `trend_pct` | Derived from same computation as label — target leakage |
| `impressions_last_30d` | Too close to label's trend computation — near-leakage |
| `clicks_last_30d` | Same as above |
| `sessions_last_30d` | Same as above |
| `impressions_prev_30d` | Same as above |
| `clicks_prev_30d` | Same as above |
| `sessions_prev_30d` | Same as above |
| `provider_used` | Data dictionary: not a model feature (internal metadata) |
| `model_used` | Data dictionary: not a model feature (internal metadata) |

## 3. Missingness audit

*Which fields are missing, how much, and is it systematic (by content_type)?*

In [2]:
# Missingness by field
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
missing_report = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print('Fields with missing values:')
print(missing_report[missing_report.missing_count > 0])

# Check if missingness is systematic by content_type
keyword_cols = ['search_volume', 'competition', 'cpc', 'competition_level']
print('\nMissingness by content_type for keyword columns:')
for col in keyword_cols:
    print(f'\n{col}:')
    print(df.groupby('content_type')[col].apply(lambda x: x.isnull().mean()).round(3))

Fields with missing values:
                   missing_count  missing_pct
provider_used              21438         71.5
word_count                  7699         25.7
char_count                  7699         25.7
word_count_tier             7699         25.7
char_count_tier             7699         25.7
model_used                  5733         19.1
trend_pct                   3388         11.3
competition_level           2610          8.7
search_volume               2468          8.2
cpc                         2468          8.2
competition                 2468          8.2
main_intent                 2374          7.9
scroll_rate                  125          0.4

Missingness by content_type for keyword columns:

search_volume:
content_type
comparison article    0.000
feedly article        1.000
keyword article       0.014
Name: search_volume, dtype: float64

competition:
content_type
comparison article    0.000
feedly article        1.000
keyword article       0.014
Name: competition,

## 4. Imputation strategy

*For each missing field group: what imputation and why?*

**Keyword columns (`search_volume`, `competition`, `cpc`, `competition_level`):**
Missing for `feedly article` rows (no keyword data). Do NOT use `fillna(0)` blindly — zero search volume would silently encode content type into the feature. Instead:
- For numeric keyword columns: fill with **median by content_type group** within training data.
- For `competition_level`: fill with `'MISSING'` category so the model can learn that feedly articles have no keyword data.
- Alternative: add a binary flag `has_keyword_data` = 1/0.

**Content length columns (`word_count`, `char_count`):**
Missing for 7,699 rows (not measured). Fill with **median by content_type** in training data, and add a `word_count_measured` flag.

**`main_intent`:** Fill with `'MISSING'` category.

**All other fields:** Should be complete after the pipeline runs `01_prepare_features.py`.